# Transmission assumptions

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/09-freq-dens-transmission.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

This chapter contrasts **frequency-dependent** and **density-dependent**
transmission. Both are expressed with summer4's
{class}`~summer4.epi.ForceOfInfection` through
{class}`~summer4.epi.ForceOfInfection` with ``kind=FOIKind.FREQUENCY`` or ``kind=FOIKind.DENSITY`` — not a hand-written
force of infection.


## Frequency dependence

Under frequency dependence the *per capita* infection rate is proportional to
the **prevalence** of infectious people. With contact rate $\beta$:

$$
\lambda(t) = \beta \, \frac{I(t)}{N(t)}
$$

Here $\beta$ is the rate at which a person makes effective contact with *any*
other member of the population. The absolute infection flow is then
$\lambda(t)\,S(t)$.

Unstratified models still need a grouping axis for the force of infection, so
we attach a dummy `pop` property with a single trait and a $1\times 1$ mixing
matrix.


In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    Everything,
    ExitFlow,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    FlowModel,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("susceptible", "infectious", "recovered"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)

RECOVERY = 0.333
END_TIME = 20.0
PLAN = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, END_TIME, int(END_TIME * 10) + 1),
)


def make_y0(population: float, seed: float) -> np.ndarray:
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["susceptible"])] = population - seed
    y0[pmap.select(state["infectious"])] = seed
    return y0


def compartment_frame(res) -> pd.DataFrame:
    """Named S/I/R columns from a saved Compartments output."""
    data = {
        name: np.asarray(res["comp"].select(state[name]).values.data)[:, 0]
        for name in ("susceptible", "infectious", "recovered")
    }
    return pd.DataFrame(data, index=np.asarray(res["comp"].times.values))


def build_sir(
    *,
    kind: str,
    contact_rate: float,
    death_rate: float = 0.0,
) -> object:
    model = FlowModel(pmap)
    
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    if kind == "frequency":
        model.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=pop,
    kind=FOIKind.FREQUENCY,
    contact_rate=contact_rate,
    mixing=mixing,
),
    )
)
    elif kind == "density":
        model.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=pop,
    kind=FOIKind.DENSITY,
    contact_rate=contact_rate,
    mixing=mixing,
),
    )
)
    else:
        raise ValueError(kind)
    model.add_flow(TransitionFlow(
        "recovery", state["infectious"], state["recovered"], RECOVERY
    ))
    if death_rate > 0.0:
        model.add_flow(ExitFlow("non_infection_deaths", Everything(), death_rate))
    return model.compile()


def run_sir(
    *,
    kind: str,
    population: float,
    seed: float,
    contact_rate: float,
    death_rate: float = 0.0,
    t1: float = END_TIME,
) -> pd.DataFrame:
    compiled = build_sir(kind=kind, contact_rate=contact_rate, death_rate=death_rate)
    y0 = make_y0(population, seed)
    res = compiled.run({}, y0, t0=0.0, t1=t1, dt=0.1, save=PLAN, solver="euler")
    return compartment_frame(res)


In [ ]:
freq_unit = run_sir(
    kind=FOIKind.FREQUENCY, population=1.0, seed=0.01, contact_rate=1.0
)
assert float(freq_unit["infectious"].max()) > 0.01
freq_unit.plot(
    title="Frequency-dependent SIR (unit population)",
    labels={"index": "time", "value": "compartment size"},
)


## Density-dependent transmission

Under density dependence the force of infection scales with the **number** of
infectious people, not their prevalence:

$$
\lambda(t) = \beta\, I(t)
$$

The symbol $\beta$ is still called `contact_rate` in the API, but its meaning
changes: it no longer includes an implicit division by $N(t)$. With a closed
population of size one the two assumptions coincide; they diverge as soon as
$N$ is not one (or not constant).


In [ ]:
dens_unit = run_sir(
    kind=FOIKind.DENSITY, population=1.0, seed=0.01, contact_rate=1.0
)
# Same unit population → identical trajectories.
assert np.allclose(freq_unit.values, dens_unit.values, atol=1e-5)
dens_unit.plot(
    title="Density-dependent SIR (unit population)",
    labels={"index": "time", "value": "compartment size"},
)


### Matching dynamics at a larger fixed population

Scale the population to 1000 and shrink the density-dependent contact rate by
the same factor. Frequency dependence keeps `contact_rate=1.0`; density
dependence uses `0.001`. The epidemic curves match again — only the
interpretation of $\beta$ differs.


In [ ]:
POP = 1000.0
SEED = 10.0

freq_large = run_sir(
    kind=FOIKind.FREQUENCY, population=POP, seed=SEED, contact_rate=1.0
)
dens_large = run_sir(
    kind=FOIKind.DENSITY, population=POP, seed=SEED, contact_rate=1.0 / POP
)
assert np.allclose(freq_large.values, dens_large.values, atol=1e-4)
dens_large.plot(
    title="Density-dependent SIR (N=1000, β scaled by 1/N)",
    labels={"index": "time", "value": "compartment size"},
)


## Doubling the population

The practical distinction appears when population size changes *without*
rescaling $\beta$. Under frequency dependence, doubling every compartment
leaves prevalence — and therefore $\lambda$ — unchanged. Under density
dependence the same doubling doubles $\lambda$, so the epidemic accelerates.


In [ ]:
freq_double = run_sir(
    kind=FOIKind.FREQUENCY, population=2.0 * POP, seed=2.0 * SEED, contact_rate=1.0
)
dens_double = run_sir(
    kind=FOIKind.DENSITY,
    population=2.0 * POP,
    seed=2.0 * SEED,
    contact_rate=1.0 / POP,  # same β as dens_large — not rescaled for 2N
)

prev_freq = freq_large["infectious"] / freq_large.sum(axis=1)
prev_freq_2 = freq_double["infectious"] / freq_double.sum(axis=1)
prev_dens = dens_large["infectious"] / dens_large.sum(axis=1)
prev_dens_2 = dens_double["infectious"] / dens_double.sum(axis=1)

assert float(np.max(np.abs(prev_freq.values - prev_freq_2.values))) < 1e-6
assert float(np.max(np.abs(prev_dens.values - prev_dens_2.values))) > 0.1

compare = pd.DataFrame(
    {
        "frequency N": prev_freq,
        "frequency 2N": prev_freq_2,
        "density N": prev_dens,
        "density 2N": prev_dens_2,
    }
)
compare.plot(
    title="Infectious prevalence after doubling population",
    labels={"index": "time", "value": "prevalence"},
)


## Changing population size over time

Deaths applied to every compartment shrink $N(t)$ during the run. Frequency
dependence still tracks prevalence, so *proportional* epidemic dynamics stay
close to the closed-population case. Density dependence does not compensate for
the shrinking denominator, so the relative final size falls and more of the
population remains susceptible.


In [ ]:
DEATH = 0.05

freq_deaths = run_sir(
    kind=FOIKind.FREQUENCY,
    population=POP,
    seed=SEED,
    contact_rate=1.0,
    death_rate=DEATH,
)
dens_deaths = run_sir(
    kind=FOIKind.DENSITY,
    population=POP,
    seed=SEED,
    contact_rate=1.0 / POP,
    death_rate=DEATH,
)

freq_deaths.plot(
    title="Frequency dependence with universal deaths",
    labels={"index": "time", "value": "compartment size"},
)


In [ ]:
freq_props = freq_deaths.div(freq_deaths.sum(axis=1), axis=0)
closed_props = freq_large.div(freq_large.sum(axis=1), axis=0)
# Per-capita trajectories stay close despite the shrinking headcount.
assert float(
    np.max(np.abs(freq_props["susceptible"] - closed_props["susceptible"]))
) < 0.02
freq_props.plot(
    title="Frequency dependence — compartment proportions",
    labels={"index": "time", "value": "proportion"},
)


In [ ]:
dens_deaths.plot(
    title="Density dependence with universal deaths",
    labels={"index": "time", "value": "compartment size"},
)


In [ ]:
dens_props = dens_deaths.div(dens_deaths.sum(axis=1), axis=0)
# Density leaves a larger susceptible fraction than the closed / frequency case.
assert float(dens_props["susceptible"].iloc[-1]) > (
    float(freq_props["susceptible"].iloc[-1]) + 0.1
)
dens_props.plot(
    title="Density dependence — compartment proportions",
    labels={"index": "time", "value": "proportion"},
)


## When to choose each assumption

If population size (or density) is changing and you expect transmission to
increase with denser crowding, density dependence is a natural choice. If
contact rates are set by behaviour that does not scale with absolute numbers —
as for many sexually transmitted infections — prefer frequency dependence.

When $N$ is fixed, either form recovers the same curves after rescaling
$\beta$; the modelling choice is then mostly about how you want to interpret
the parameter.

## Summary

| | Frequency dependence | Density dependence |
|---|---|---|
| Quantity scaling $\lambda$ | Infectious prevalence $I/N$ | Infectious count $I$ |
| $\lambda(t)$ | $\beta\,I(t)/N(t)$ | $\beta\,I(t)$ |
| Doubling a closed population | Prevalence trajectory unchanged | Epidemic accelerates |
| Changing $N(t)$ mid-epidemic | Proportional dynamics stable | Final size / remaining $S$ shift |
